In [ ]:
# ==============================================================================
# ⚡ Cloud Ultra Downloader - Single Unified Cell (1-Click Run)
# Repo: https://github.com/ElkampaPro/cloud-ultra-downloader
# ==============================================================================

import os
import re
import time
import subprocess
from pathlib import Path
from IPython.display import HTML, display

print("=" * 70)
print("🚀 بدء تشغيل منصة Cloud Ultra Downloader السحابية...")
print("=" * 70)

# 1. Mount Google Drive (5TB Storage)
print("\n[1/5] ☁️ جارٍ ربط Google Drive لحفظ الملفات مباشرة...")
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"⚠️ تنبيه درايف: {e}")

DOWNLOAD_DIR = "/content/drive/MyDrive/Downloads"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
print(f"✅ مجلد التنزيلات جاهز: {DOWNLOAD_DIR}")

# 2. Install Aria2c & Tools
print("\n[2/5] 📦 تثبيت محرك Aria2c والحزم المطلوبة...")
subprocess.run("apt-get update -qq && apt-get install -y -qq aria2 wget curl git", shell=True, check=True)

# 3. Clone or Update Project
print("\n[3/5] 📥 جلب أحدث كود لمشروع Cloud Ultra Downloader...")
PROJECT_DIR = "/content/cloud-ultra-downloader"
REPO_URL = "https://github.com/ElkampaPro/cloud-ultra-downloader.git"

if not os.path.exists(PROJECT_DIR):
    subprocess.run(f"git clone {REPO_URL} {PROJECT_DIR}", shell=True, check=True)
else:
    subprocess.run(f"cd {PROJECT_DIR} && git pull", shell=True, check=True)

subprocess.run(f"pip install -q -r {PROJECT_DIR}/requirements.txt", shell=True, check=True)

# 4. Clean previous instances
subprocess.run("pkill -f aria2c || true", shell=True)
subprocess.run("pkill -f uvicorn || true", shell=True)
subprocess.run("pkill -f cloudflared || true", shell=True)
time.sleep(1)

# 5. Start Aria2c Daemon (16 Multi-stream acceleration)
print("\n[4/5] ⚡ تشغيل محرك Aria2c بـ 16 مسار اتصال متوازي...")
aria2_cmd = [
    "aria2c",
    "--enable-rpc",
    "--rpc-listen-all=true",
    "--rpc-listen-port=6800",
    "--max-connection-per-server=16",
    "--split=16",
    "--min-split-size=1M",
    "--max-concurrent-downloads=5",
    "--continue=true",
    f"--dir={DOWNLOAD_DIR}",
    "--rpc-allow-origin-all=true",
    "--header=User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "--header=Accept: */*",
    "--daemon=true"
]
subprocess.run(aria2_cmd, check=True)
time.sleep(1)
print("✅ محرك Aria2c يعمل الآن بنجاح.")

# 6. Start FastAPI Web Server in Background
print("\n[5/5] 🌐 تشغيل خادم الواجهة ونفق Cloudflare المشفر...")
web_proc = subprocess.Popen(
    ["python3", "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=PROJECT_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(2)

# 7. Start Cloudflare Tunnel
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared", shell=True, check=True)

cf_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = cf_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.5)

if tunnel_url:
    print("\n" + "=" * 70)
    print("🎉 تم تشغيل لوحة التحكم السحابية بنجاح!")
    print(f"🔗 الرابط المباشر: {tunnel_url}")
    print("=" * 70)
    
    html_card = f'''
    <div style="padding: 24px; background: linear-gradient(135deg, #090d16 0%, #1e1b4b 100%); border-radius: 18px; border: 1px solid #4338ca; color: white; font-family: 'Cairo', sans-serif; text-align: center; margin-top: 15px; box-shadow: 0 10px 25px rgba(0,0,0,0.5);">
        <h2 style="margin: 0 0 10px 0; color: #38bdf8; font-size: 22px;">⚡ لوحة تحكم Cloud Ultra Downloader جاهزة!</h2>
        <p style="margin: 0 0 16px 0; font-size: 14px; color: #cbd5e1;">انقر على الزر أدناه لفتح الواجهة من هاتفك أو حاسوبك وبدء التحميل إلى Google Drive (5TB):</p>
        <a href="{tunnel_url}" target="_blank" style="display: inline-block; padding: 14px 34px; background: linear-gradient(90deg, #6366f1, #4f46e5); color: white; text-decoration: none; border-radius: 12px; font-weight: bold; font-size: 17px; box-shadow: 0 4px 20px rgba(99, 102, 241, 0.5);">🚀 افتح لوحة التحكم (Open Web Dashboard)</a>
        <p style="margin: 16px 0 0 0; font-size: 13px; color: #94a3b8; font-family: monospace;">{tunnel_url}</p>
    </div>
    '''
    display(HTML(html_card))
    
    try:
        while True:
            time.sleep(3600)
    except KeyboardInterrupt:
        print("\nتم إيقاف السيرفر.")
else:
    print("⚠️ تعذر التقاط رابط Cloudflare تلقائياً.")
